In this project, rather than depending on pytorch's prebuilt backward() function, we will manually construct the backward propogation in order to fully understand how it works. We will be working on the same model and dataset as the previous projects.

In [ ]:
import torch, torch.nn.functional as F, matplotlib.pyplot as plt
import random

#Loading data and setting up data split

words = open('names.txt', 'r').read().splitlines()

chars = sorted(list(set(''.join(words))))
stoi = {s: i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}

print(f"Characters: \n",chars)
print(f"integer to string: \n",itos)
print(f"String to integer: \n",stoi)

random.seed(42)
random.shuffle(words)

n1=int(len(words)*0.8)
n2=int(len(words)*0.9)

train_words = words[:n1]
val_words = words[n1:n2]
test_words=words[n2:]

def build_dataset(words):

    block_size = 3
    X, Y = [], []

    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]

    X=torch.tensor(X)
    Y=torch.tensor(Y)

    return X,Y

Xtr,Ytr = build_dataset(train_words)
Xval,Yval = build_dataset(val_words)
Xtest,Ytest = build_dataset(test_words)


print(f"\nX training Shape:",Xtr.shape)
print(f" \nX Validation Shape:", Xval.shape)
print(f" \nX Test Shape:", Xtest.shape)


#Building MLP
g = torch.Generator().manual_seed(2147483647)
n_hidden = 64
n_embedding = 10
block_size = 3

C = torch.randn((27, n_embedding), generator=g)

#Layer 1
W1=torch.randn((n_embedding * block_size, n_hidden), generator=g) * (5/3)/((n_embedding * block_size) ** 0.5)
b1=torch.randn(n_hidden, generator=g) * 0.1

#Layer 2
W2 = torch.randn((n_hidden, 27), generator=g) * 0.1
b2 = torch.zeros(27, generator=g) * 0.1

#Batchnorm Parameters
bngain = torch.randn((1, n_hidden))*0.1 + 1.0
bnbias = torch.randn((1, n_hidden))*0.1

parameters = [C, W1, b1, W2, b2, bngain]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True


Now, we will conduct the forward pass. We will split it up into the finest steps

In [ ]:
#First, create the batches

batch_size = 32
n = batch_size
ix = torch.randint(0, Xtr.shape[0], (batch_size, ), generator=g)
Xb,Yb = Xtr[ix], Ytr[ix]

#Now, the forward pass

emb = C[Xb] #Embed the characters
embcat = emb.view(emb.shape[0], -1)#Concatenate the vector
#Linear Layer 1
hprebn = embcat @ W1 + b1 #Hidden layer pre activation
#Batchnorm Layer
bnmeani = (1 / n) * hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff ** 2
bnvar = (1 / (n-1) ) * bndiff2.sum(0, keepdim=True)#Bessels correction (dividing by n-1 not n)
bnvar_inv = (bnvar + 1e-5) ** -0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias
#Non-Linearity
h = torch.tanh(hpreact)
#Linear Layer 2
logits = hpreact @ W1 + b1
# cross entropy loss (same as F.cross_entropy(logits, Yb))
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes # subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()